In [ ]:
!pip install requests trafilatura beautifulsoup4 langchain langgraph groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 36.1 MB/s eta 0:00:00


In [ ]:
import requests
import json
from datetime import datetime, timedelta
import gradio as gr

In [ ]:
COMPETITION_TYPES = {
    "Big 5 Leagues": {
        "Premier League": "eng.1",
        "La Liga": "esp.1",
        "Serie A": "ita.1",
        "Bundesliga": "ger.1",
        "Ligue 1": "fra.1"
    },
    "European Tournaments": {
        "Champions League": "uefa.champions"
    }
}

ARTICLE_FOCUSES = [
    "Full Match Report",
    "Star Player Performance",
    "Key Match Moments",
    "Tactical Analysis",
    "Timeline Summary",
    "Short News Summary"
]

In [ ]:
def get_competitions(competition_type):
    return list(COMPETITION_TYPES[competition_type].keys())


def get_league_code(competition_type, competition_name):
    return COMPETITION_TYPES[competition_type][competition_name]

In [ ]:
def get_latest_4_matches(league_code, days_back=30):
    headers = {"User-Agent": "Mozilla/5.0"}
    matches = []

    today = datetime.today()

    for i in range(days_back):
        date = today - timedelta(days=i)
        date_str = date.strftime("%Y%m%d")

        url = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{league_code}/scoreboard?dates={date_str}"

        try:
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            data = response.json()

            for event in data.get("events", []):
                competition = event["competitions"][0]
                status = competition["status"]["type"]

                if not status.get("completed"):
                    continue

                competitors = competition.get("competitors", [])

                home = next((c for c in competitors if c.get("homeAway") == "home"), None)
                away = next((c for c in competitors if c.get("homeAway") == "away"), None)

                if not home or not away:
                    continue

                matches.append({
                    "event_id": event["id"],
                    "league": league_code,
                    "date": event.get("date"),
                    "home_team": home["team"]["displayName"],
                    "away_team": away["team"]["displayName"],
                    "home_score": home.get("score"),
                    "away_score": away.get("score"),
                    "display": f'{away["team"]["displayName"]} {away.get("score")} - {home.get("score")} {home["team"]["displayName"]}'
                })

            if len(matches) >= 4:
                break

        except Exception as e:
            print("Error:", e)
            continue

    return matches[:4]

In [ ]:
def get_match_details(event_id, league_code):
    url = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{league_code}/summary?event={event_id}"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code != 200:
        raise Exception(f"Request failed with status code {response.status_code}")

    data = response.json()

    header = data.get("header", {})
    competition = header.get("competitions", [{}])[0]
    competitors = competition.get("competitors", [])

    teams = []

    for c in competitors:
        teams.append({
            "team": c["team"]["displayName"],
            "home_away": c.get("homeAway"),
            "score": c.get("score")
        })

    key_events = []

    for e in data.get("keyEvents", []):
        key_events.append({
            "time": e.get("clock", {}).get("displayValue"),
            "type": e.get("type", {}).get("text"),
            "description": e.get("text")
        })

    key_events = clean_events(key_events)

    match_details = {
        "event_id": event_id,
        "league": league_code,
        "status": competition.get("status", {}).get("type", {}).get("description"),
        "teams": teams,
        "key_events": key_events,
    }

    return match_details

def clean_events(events):
    cleaned = []

    for e in events:
        if e["description"] is None:
            continue

        cleaned.append(e)

    return cleaned

In [ ]:
def prepare_article_context(match_details, focus):
    teams = match_details["teams"]
    key_events = match_details["key_events"]

    home = next((t for t in teams if t["home_away"] == "home"), None)
    away = next((t for t in teams if t["home_away"] == "away"), None)

    goals = [e for e in key_events if e["type"] == "Goal"]
    cards = [e for e in key_events if e["type"] == "Yellow Card"]
    substitutions = [e for e in key_events if e["type"] == "Substitution"]

    context = {
        "match": f"{home['team']} {home['score']} - {away['score']} {away['team']}",
        "status": match_details["status"],
        "focus": focus,
        "home_team": home,
        "away_team": away,
        "goals": goals,
        "cards": cards,
        "substitutions": substitutions,
        "key_events": key_events,
    }

    return context

In [ ]:
def detect_standout_players(article_context):
    standout_players = []

    for goal in article_context["goals"]:
        description = goal["description"]

        if "(" in description and ")" in description:
            player_part = description.split(".")[1].strip()
            player_name = player_part.split("(")[0].strip()

            standout_players.append({
                "name": player_name,
                "reason": f"Scored a goal at {goal['time']}"
            })

    return standout_players

In [ ]:
def get_league_context(league_code):
    url = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{league_code}/standings"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code != 200:
        print("Standings request failed:", response.status_code)
        return []

    data = response.json()
    table = []

    groups = data.get("children", [])

    for group in groups:
        standings = group.get("standings", {})
        entries = standings.get("entries", [])

        for entry in entries:
            team = entry.get("team", {})
            stats = entry.get("stats", [])

            position = None
            points = None

            for stat in stats:
                if stat.get("name") in ["rank", "position"]:
                    position = stat.get("displayValue") or stat.get("value")
                if stat.get("name") in ["points", "pts"]:
                    points = stat.get("displayValue") or stat.get("value")

            table.append({
                "team": team.get("displayName"),
                "short_name": team.get("shortDisplayName"),
                "abbreviation": team.get("abbreviation"),
                "position": position,
                "points": points
            })

    return table

In [ ]:
def find_team_in_table(team_name, table):
    team_name_lower = team_name.lower()

    for team in table:
        names = [
            team.get("team", ""),
            team.get("short_name", ""),
            team.get("abbreviation", "")
        ]

        for name in names:
            if name and (team_name_lower in name.lower() or name.lower() in team_name_lower):
                return team

    return None

In [ ]:
def get_auto_aggregate_context(selected_match, match_details, days_window=30):
    league_code = selected_match["league"]

    # only do this for Champions League
    if league_code != "uefa.champions":
        return None

    teams = match_details["teams"]
    team_names = [t["team"] for t in teams]

    selected_event_id = str(selected_match["event_id"])

    selected_date = datetime.fromisoformat(
        selected_match["date"].replace("Z", "+00:00")
    )

    related_matches = []

    headers = {"User-Agent": "Mozilla/5.0"}

    for i in range(-days_window, days_window + 1):
        date = selected_date + timedelta(days=i)
        date_str = date.strftime("%Y%m%d")

        url = f"https://site.api.espn.com/apis/site/v2/sports/soccer/{league_code}/scoreboard?dates={date_str}"

        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                continue

            data = response.json()

            for event in data.get("events", []):
                competition = event["competitions"][0]
                status = competition["status"]["type"]

                if not status.get("completed"):
                    continue

                competitors = competition.get("competitors", [])
                event_team_names = [c["team"]["displayName"] for c in competitors]

                same_two_teams = all(team in event_team_names for team in team_names)

                if same_two_teams:
                    related_matches.append(event)

        except Exception:
            continue

    unique = {}
    for match in related_matches:
        unique[str(match["id"])] = match

    related_matches = list(unique.values())

    if len(related_matches) < 2:
        return None

    aggregate = {team_names[0]: 0, team_names[1]: 0}
    leg_results = []

    for event in related_matches:
        competition = event["competitions"][0]
        competitors = competition["competitors"]

        leg_score = {}

        for c in competitors:
            team = c["team"]["displayName"]
            score = int(c.get("score", 0))
            leg_score[team] = score

            if team in aggregate:
                aggregate[team] += score

        leg_results.append({
            "event_id": event["id"],
            "date": event.get("date"),
            "score": leg_score
        })

    team_a, team_b = team_names
    score_a = aggregate[team_a]
    score_b = aggregate[team_b]

    if score_a > score_b:
        winner = team_a
    elif score_b > score_a:
        winner = team_b
    else:
        winner = "Tie level"

    return {
        "is_knockout": True,
        "legs_found": len(related_matches),
        "leg_results": leg_results,
        "aggregate_score": f"{team_a} {score_a} - {score_b} {team_b}",
        "winner_on_aggregate": winner,
        "tie_outcome": (
            f"{winner} advanced {score_a}-{score_b} on aggregate"
            if winner != "Tie level"
            else f"The tie finished level {score_a}-{score_b} on aggregate"
        )
    }

In [ ]:
def add_knockout_context(article_context, competition_name, selected_match=None, match_details=None):
    if competition_name == "Champions League":
        article_context["competition_type"] = "knockout"

        if selected_match is not None and match_details is not None:
            aggregate_context = get_auto_aggregate_context(selected_match, match_details)
        else:
            aggregate_context = None

        article_context["knockout_context"] = aggregate_context
    else:
        article_context["competition_type"] = "league"
        article_context["knockout_context"] = None

    return article_context

In [ ]:
import os
from getpass import getpass
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])



Enter your Groq API key: ··········


In [ ]:
def generate_article_with_groq(article_context):
    prompt = f"""
  You are an elite football journalist writing for BBC Sport or ESPN.

  Write a high-quality, professional match report using ONLY the facts provided.
  Do NOT invent any players, goals, events, standings, or context.

  Your writing MUST:
  - Read like a real sports article (not a list)
  - Tell the story of the match chronologically
  - Add drama, flow, and narrative
  - Emphasize momentum shifts and turning points
  - Clearly highlight standout players
  - Use the standout_players field when writing the player highlights section
  - Be concise but rich

  Avoid:
  - listing substitutions one by one
  - repeating the same information
  - robotic phrasing

  Article focus: {article_context["focus"]}

  Competition awareness rules:
   Mention the position of the leauge table both teams are in and what the result meant for each if its a leauge match
   If its a european tournament - knockout game- mention the aggregate and what the results meant for each


  Structure:
  1. HEADLINE (strong and engaging)
  2. INTRO (include key player and decisive moment)
  3. MATCH NARRATIVE
  4. KEY MOMENTS
  5. PLAYER HIGHLIGHTS (use standout_players)
  6. CONCLUSION (If)

  If knockout_context is provided:
  - You MUST mention the aggregate_score.
  - You MUST clearly state winner_on_aggregate.
  - If the match result and aggregate result differ, explain that clearly.

If knockout_context is None:
  - Do NOT mention aggregate score.
  - Do NOT say who advanced.
    In the conclusion:
    - Briefly explain the significance of the result
    - Keep it sharp and avoid generic future speculation

  Match data:
  {json.dumps(article_context, indent=2, ensure_ascii=False)}
  """

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You write accurate, professional football match reports in a BBC Sport / ESPN style."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.6
    )

    return chat_completion.choices[0].message.content

In [ ]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END

class FootballAgentState(TypedDict):
    competition_type: str
    competition_name: str
    league_code: str
    matches: List[Dict[str, Any]]
    selected_match_index: int
    selected_match: Dict[str, Any]
    match_details: Dict[str, Any]
    focus: str
    article_context: Dict[str, Any]
    standout_players: List[Dict[str, str]]
    article: str

In [ ]:
def select_league_node(state):
    league_code = get_league_code(
        state["competition_type"],
        state["competition_name"]
    )

    state["league_code"] = league_code
    return state


def load_matches_node(state):
    matches = get_latest_4_matches(state["league_code"])
    state["matches"] = matches
    return state


def select_match_node(state):
    index = state["selected_match_index"]
    state["selected_match"] = state["matches"][index]
    return state


def fetch_match_details_node(state):
    selected = state["selected_match"]

    match_details = get_match_details(
        selected["event_id"],
        selected["league"]
    )

    state["match_details"] = match_details
    return state


def prepare_context_node(state):
    article_context = prepare_article_context(
        state["match_details"],
        state["focus"]
    )

    state["article_context"] = article_context
    return state


def standout_players_node(state):
    standout_players = detect_standout_players(state["article_context"])

    state["standout_players"] = standout_players
    state["article_context"]["standout_players"] = standout_players

    return state


def generate_article_node(state):
    article = generate_article_with_groq(state["article_context"])
    state["article"] = article
    return state

In [ ]:
def add_league_context_node(state):
    table = get_league_context(state["league_code"])

    home = state["article_context"]["home_team"]["team"]
    away = state["article_context"]["away_team"]["team"]

    home_info = next((t for t in table if t["team"] == home), None)
    away_info = next((t for t in table if t["team"] == away), None)

    state["article_context"]["league_context"] = {
        "home_team": home_info,
        "away_team": away_info
    }

    return state

In [ ]:
def add_knockout_context_node(state):
    article_context = add_knockout_context(
        state["article_context"],
        state["competition_name"]
    )

    state["article_context"] = article_context
    return state

In [ ]:
def build_football_graph():
    graph = StateGraph(FootballAgentState)

    graph.add_node("select_league", select_league_node)
    graph.add_node("load_matches", load_matches_node)
    graph.add_node("select_match", select_match_node)
    graph.add_node("fetch_match_details", fetch_match_details_node)
    graph.add_node("prepare_context", prepare_context_node)
    graph.add_node("detect_standout_players", standout_players_node)
    graph.add_node("add_league_context", add_league_context_node)
    graph.add_node("add_knockout_context",add_knockout_context)
    graph.add_node("generate_article", generate_article_node)


    graph.set_entry_point("select_league")

    graph.add_edge("select_league", "load_matches")
    graph.add_edge("load_matches", "select_match")
    graph.add_edge("select_match", "fetch_match_details")
    graph.add_edge("fetch_match_details", "prepare_context")
    graph.add_edge("prepare_context", "add_knockout_context")
    graph.add_edge("add_knockout_context", "add_league_context")
    graph.add_edge("add_league_context", "detect_standout_players")
    graph.add_edge("detect_standout_players", "generate_article")
    graph.add_edge("generate_article", END)

    return graph.compile()

In [ ]:
football_graph = build_football_graph()

In [ ]:
import gradio as gr

latest_matches_cache = {}

def update_competitions(competition_type):
    competitions = list(COMPETITION_TYPES[competition_type].keys())
    return gr.update(choices=competitions, value=competitions[0])


def load_latest_matches(competition_type, competition_name):
    league_code = get_league_code(competition_type, competition_name)
    matches = get_latest_4_matches(league_code)

    latest_matches_cache.clear()

    choices = []

    for match in matches:
        label = f"{match['display']} | ID: {match['event_id']}"
        latest_matches_cache[label] = match
        choices.append(label)

    if not choices:
        return gr.update(choices=[], value=None), "No completed matches found."

    return gr.update(choices=choices, value=choices[0]), "Latest 4 matches loaded."


def run_pipeline_from_match(competition_type, competition_name, selected_match_label, focus):
    if not selected_match_label:
        return "Please load and select a match first."

    selected_match = latest_matches_cache[selected_match_label]

    # 1. Get match data
    match_details = get_match_details(
        selected_match["event_id"],
        selected_match["league"]
    )

    # 2. Build base context
    article_context = prepare_article_context(match_details, focus)

    # 3. Add knockout context (Champions League fix)
    article_context = add_knockout_context(
    article_context,
    competition_name,
    selected_match=selected_match,
    match_details=match_details
)

    # 4. Add league table context ONLY for league competitions
    if competition_type == "Big 5 Leagues":
        table = get_league_context(selected_match["league"])

        home = article_context["home_team"]["team"]
        away = article_context["away_team"]["team"]

        home_info = find_team_in_table(home, table)
        away_info = find_team_in_table(away, table)

        article_context["league_context"] = {
            "home_team": home_info,
            "away_team": away_info
        }
    else:
        article_context["league_context"] = None

    # 5. Add standout players
    standout_players = detect_standout_players(article_context)
    article_context["standout_players"] = standout_players

    # 6. Generate article
    article = generate_article_with_groq(article_context)

    return article


with gr.Blocks(title="Autonomous Football Journalist") as demo:
    gr.Markdown("#  Autonomous Football Journalist")
    gr.Markdown("Generate professional football match reports using ESPN data, LangGraph, and Groq.")

    with gr.Row():
        competition_type = gr.Dropdown(
            choices=list(COMPETITION_TYPES.keys()),
            value="Big 5 Leagues",
            label="Competition Category"
        )

        competition_name = gr.Dropdown(
            choices=list(COMPETITION_TYPES["Big 5 Leagues"].keys()),
            value="Premier League",
            label="League / Tournament"
        )

    load_btn = gr.Button("Load Latest 4 Completed Matches")

    match_dropdown = gr.Dropdown(
        choices=[],
        label="Select Match"
    )

    focus = gr.Dropdown(
        choices=ARTICLE_FOCUSES,
        value="Full Match Report",
        label="Article Focus"
    )

    generate_btn = gr.Button("Generate Article")

    status_box = gr.Textbox(label="Status", interactive=False)

    output = gr.Textbox(
        label="Generated Article",
        lines=20
    )

    competition_type.change(
        update_competitions,
        inputs=competition_type,
        outputs=competition_name
    )

    load_btn.click(
        load_latest_matches,
        inputs=[competition_type, competition_name],
        outputs=[match_dropdown, status_box]
    )

    generate_btn.click(
        run_pipeline_from_match,
        inputs=[competition_type, competition_name, match_dropdown, focus],
        outputs=output
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9f9aec47eabee7fcb1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
